In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("fraud-streaming") \
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.1.0,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("✅ Spark session ready")

In [2]:
from pyspark.sql.functions import from_json, col, window, sum as _sum, count, avg
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("type", StringType()),
    StructField("amount", DoubleType()),
    StructField("nameOrig", StringType()),
    StructField("nameDest", StringType()),
    StructField("oldbalanceOrg", DoubleType()),
    StructField("newbalanceOrig", DoubleType()),
    StructField("isFraud", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("source", StringType()),
])

raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw.select(
    from_json(col("value").cast("string"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

print("✅ Streaming reader defined")

In [3]:
checkpoint_path = "s3a://silver/checkpoints/streaming/"
output_path = "s3a://silver/streaming/transactions/"

query = parsed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .option("path", output_path) \
    .trigger(processingTime="10 seconds") \
    .start()

print("✅ Streaming query started — writing to silver Delta table")
print(f"Status: {query.status}")

In [6]:
print(query.status)
print(query.lastProgress)

In [8]:
print(query.status)
print(query.lastProgress)

In [9]:
query.stop()

In [10]:
raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw.select(
    from_json(col("value").cast("string"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

query = parsed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/streaming/") \
    .option("path", "s3a://silver/streaming/transactions/") \
    .trigger(processingTime="10 seconds") \
    .start()

print(f"✅ Stream started — status: {query.status}")

In [11]:
print(query.status)
print(query.lastProgress)

In [12]:
print(query.status)
print(query.lastProgress)

In [13]:
query.stop()

In [14]:
raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw.select(
    from_json(col("value").cast("string"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

query = parsed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/streaming/") \
    .option("path", "s3a://silver/streaming/transactions/") \
    .trigger(processingTime="10 seconds") \
    .start()

print(f"✅ Stream started — status: {query.status}")

In [15]:
print(query.status)
print(query.lastProgress)

In [16]:
df = spark.read.format("delta").load("s3a://silver/streaming/transactions/")
print(f"✅ Rows in silver streaming table: {df.count()}")
df.show(5)

In [17]:
from pyspark.sql.functions import to_timestamp, window, col, sum as _sum, count, avg

# Stop current query first
query.stop()

# Re-read stream with watermark
raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw.select(
    from_json(col("value").cast("string"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

# Add proper timestamp and watermark for late data (10 minute tolerance)
with_watermark = parsed \
    .withColumn("event_time", to_timestamp(col("timestamp"))) \
    .withWatermark("event_time", "10 minutes")

# Sliding window aggregations per card per 1h window
windowed = with_watermark.groupBy(
    window(col("event_time"), "1 hour", "10 minutes"),
    col("nameOrig")
).agg(
    count("transaction_id").alias("tx_count_1h"),
    _sum("amount").alias("total_amount_1h"),
    avg("amount").alias("avg_amount_1h"),
    _sum("isFraud").alias("fraud_count_1h")
)

query_windowed = windowed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/windowed/") \
    .option("path", "s3a://silver/streaming/windowed_features/") \
    .trigger(processingTime="10 seconds") \
    .start()

print(f"✅ Windowed stream started: {query_windowed.status}")

In [18]:
df_w = spark.read.format("delta").load("s3a://silver/streaming/windowed_features/")
print(f"Windowed features rows: {df_w.count()}")
df_w.show(5, truncate=False)

In [19]:
df_w = spark.read.format("delta").load("s3a://silver/streaming/windowed_features/")
print(f"Windowed features rows: {df_w.count()}")
df_w.show(5, truncate=False)

In [20]:
import boto3
s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket="silver", Prefix="checkpoints/windowed/"):
    for obj in page.get("Contents", []):
        s3.delete_object(Bucket="silver", Key=obj["Key"])
print("✅ Checkpoint cleared")

In [21]:
query_windowed.stop()

In [24]:
query_windowed = windowed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/windowed/") \
    .option("path", "s3a://silver/streaming/windowed_features/") \
    .trigger(processingTime="10 seconds") \
    .start()

In [25]:
df_check = spark.read.format("delta") \
    .load("s3a://silver/streaming/windowed_features/")

df_check.show(truncate=False)

In [26]:
import boto3
s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket="silver", Prefix="checkpoints/windowed/"):
    for obj in page.get("Contents", []):
        s3.delete_object(Bucket="silver", Key=obj["Key"])
print("✅ Checkpoint cleared")

In [27]:
query_windowed.stop()

query_windowed = windowed.writeStream \
    .format("delta") \
    .outputMode("update") \
    .option("checkpointLocation", "s3a://silver/checkpoints/windowed/") \
    .option("path", "s3a://silver/streaming/windowed_features/") \
    .trigger(processingTime="10 seconds") \
    .start()
print(query_windowed.status)

In [28]:
query_windowed.stop()

# Clear checkpoint
import boto3
s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket="silver", Prefix="checkpoints/windowed/"):
    for obj in page.get("Contents", []):
        s3.delete_object(Bucket="silver", Key=obj["Key"])
print("✅ Checkpoint cleared")

# Redefine with SHORT window for testing
windowed = with_watermark.groupBy(
    window(col("event_time"), "1 minute", "30 seconds"),
    col("nameOrig")
).agg(
    count("transaction_id").alias("tx_count_1h"),
    _sum("amount").alias("total_amount_1h"),
    avg("amount").alias("avg_amount_1h"),
    _sum("isFraud").alias("fraud_count_1h")
)

query_windowed = windowed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/windowed/") \
    .option("path", "s3a://silver/streaming/windowed_features/") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✅ Windowed stream started:", query_windowed.status)

In [29]:
df_w = spark.read.format("delta").load("s3a://silver/streaming/windowed_features/")
print(f"Rows: {df_w.count()}")
df_w.show(5, truncate=False)

In [30]:
print(query_windowed.status)
print(query_windowed.lastProgress)

In [31]:
df_w = spark.read.format("delta").load("s3a://silver/streaming/windowed_features/")
print(f"Rows: {df_w.count()}")
df_w.show(5, truncate=False)

In [32]:
query_windowed.stop()

query_windowed_mem = windowed.writeStream \
    .format("memory") \
    .queryName("windowed_check") \
    .outputMode("complete") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✅ Memory stream started:", query_windowed_mem.status)

In [33]:
spark.sql("SELECT * FROM windowed_check ORDER BY window DESC LIMIT 10").show(truncate=False)

In [34]:
query_windowed_mem.stop()
query_windowed.stop()

# Clear checkpoint
import boto3
s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket="silver", Prefix="checkpoints/windowed/"):
    for obj in page.get("Contents", []):
        s3.delete_object(Bucket="silver", Key=obj["Key"])
print("✅ Checkpoint cleared")

# Redefine with short watermark
with_watermark = parsed \
    .withColumn("event_time", to_timestamp(col("timestamp"))) \
    .withWatermark("event_time", "30 seconds")  # ← was 10 minutes

windowed = with_watermark.groupBy(
    window(col("event_time"), "1 minute", "30 seconds"),
    col("nameOrig")
).agg(
    count("transaction_id").alias("tx_count_1h"),
    _sum("amount").alias("total_amount_1h"),
    avg("amount").alias("avg_amount_1h"),
    _sum("isFraud").alias("fraud_count_1h")
)

query_windowed = windowed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/windowed/") \
    .option("path", "s3a://silver/streaming/windowed_features/") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✅ Stream started:", query_windowed.status)

In [35]:
df_w = spark.read.format("delta").load("s3a://silver/streaming/windowed_features/")
print(f"Rows: {df_w.count()}")
df_w.show(5, truncate=False)

In [38]:
import subprocess
subprocess.run(["pip", "install", "confluent-kafka"], capture_output=True)

from confluent_kafka import Producer
import json

def send_fraud_alerts(batch_df, batch_id):
    fraud = batch_df.filter(col("isFraud") == 1)
    count = fraud.count()
    if count > 0:
        producer = Producer({"bootstrap.servers": "kafka:29092"})
        for row in fraud.collect():
            producer.produce(
                "fraud-alerts",
                value=json.dumps({
                    "transaction_id": row["transaction_id"],
                    "amount": row["amount"],
                    "nameOrig": row["nameOrig"],
                    "nameDest": row["nameDest"],
                    "type": row["type"],
                    "timestamp": str(row["timestamp"])
                }).encode("utf-8")
            )
        producer.flush()
        print(f"🚨 Batch {batch_id}: sent {count} fraud alerts")

query_alerts = parsed.writeStream \
    .foreachBatch(send_fraud_alerts) \
    .option("checkpointLocation", "s3a://silver/checkpoints/fraud_alerts/") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✅ Fraud alerts stream started:", query_alerts.status)

In [39]:
from confluent_kafka import Consumer
import json

consumer = Consumer({
    "bootstrap.servers": "kafka:29092",
    "group.id": "fraud-alert-verifier",
    "auto.offset.reset": "earliest"
})
consumer.subscribe(["fraud-alerts"])

print("Reading fraud alerts from Kafka...")
alerts = []
for _ in range(30):
    msg = consumer.poll(timeout=1.0)
    if msg and not msg.error():
        alerts.append(json.loads(msg.value()))

consumer.close()
print(f"✅ Found {len(alerts)} fraud alerts in topic")
for a in alerts[:3]:
    print(a)

In [40]:
from confluent_kafka.admin import AdminClient
from confluent_kafka import Consumer, TopicPartition

def get_kafka_lag(topic, group_id):
    consumer = Consumer({
        "bootstrap.servers": "kafka:29092",
        "group.id": group_id,
        "auto.offset.reset": "earliest"
    })
    admin = AdminClient({"bootstrap.servers": "kafka:29092"})
    
    metadata = admin.list_topics(topic=topic)
    partitions = [TopicPartition(topic, p) for p in metadata.topics[topic].partitions]
    
    committed = consumer.committed(partitions)
    end_offsets = consumer.get_watermark_offsets
    
    total_lag = 0
    for tp in committed:
        low, high = consumer.get_watermark_offsets(tp)
        committed_offset = tp.offset if tp.offset >= 0 else low
        lag = high - committed_offset
        total_lag += lag
        print(f"  Partition {tp.partition}: committed={committed_offset}, latest={high}, lag={lag}")
    
    consumer.close()
    print(f"Total lag for '{topic}': {total_lag}")

print("=== transactions topic ===")
get_kafka_lag("transactions", "spark-streaming-group")

print("\n=== fraud-alerts topic ===")
get_kafka_lag("fraud-alerts", "fraud-alert-verifier")